# Ablation Study: Polar vs DETR vs Autoregressive Decoder

**PhD Thesis — Diana Paola Ayala Roldan**

CNN-Transformer with polar-parameterized decoder for autonomous wound boundary detection.

This notebook runs the complete ablation study on Kaggle GPU:
1. Generate synthetic dataset (2,000 samples)
2. Train all 3 decoder variants with identical encoder + hyperparameters
3. Evaluate on test set
4. Generate figures for thesis

**Runtime:** ~2-3 hours on Kaggle P100 GPU

In [ ]:
# Setup: install missing deps if on Kaggle
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'PuLP', 'spatialmath-python'])

In [ ]:
import sys, os
sys.path.insert(0, '/kaggle/working/diana-bioprinting-pipeline')  # Kaggle
sys.path.insert(0, '..')  # Local

import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 1. Generate Synthetic Dataset

In [ ]:
from data.synthetic_generator import generate_dataset

SYNTHETIC_DIR = 'data/synthetic'
NUM_SYNTHETIC = 2000

# Only generate if not already done
if not Path(SYNTHETIC_DIR, 'labels.npz').exists():
    print('Generating synthetic dataset...')
    generate_dataset(SYNTHETIC_DIR, num_samples=NUM_SYNTHETIC, seed=42)
else:
    print(f'Synthetic dataset already exists at {SYNTHETIC_DIR}')

## 2. Visualize Samples

In [ ]:
from data.synthetic_generator import generate_star_convex_wound
from data.polar_conversion import polar_to_cartesian

fig, axes = plt.subplots(2, 4, figsize=(14, 7))

for i in range(4):
    np.random.seed(i * 7 + 1)
    sample = generate_star_convex_wound()
    
    # Image with boundary overlay
    axes[0, i].imshow(sample['image'])
    pts = sample['points'] * 256
    pts_closed = np.vstack([pts, pts[0]])
    axes[0, i].plot(pts_closed[:, 0], pts_closed[:, 1], 'lime', linewidth=2)
    axes[0, i].plot(sample['centroid'][0]*256, sample['centroid'][1]*256, 'r+', markersize=12)
    axes[0, i].set_title(f'Sample {i+1}')
    axes[0, i].axis('off')
    
    # Polar plot (radii vs angle)
    angles = np.linspace(0, 360, len(sample['radii']), endpoint=False)
    axes[1, i].bar(angles, sample['radii'] * 256, width=5, color='coral', alpha=0.7)
    axes[1, i].set_xlabel('Angle (deg)')
    axes[1, i].set_ylabel('Radius (px)')
    axes[1, i].set_title(f'Polar repr.')

plt.suptitle('Synthetic Wound Samples + Polar Ground Truth', fontsize=14)
plt.tight_layout()
plt.savefig('figures/synthetic_samples.png', dpi=150)
plt.show()

## 3. Run Ablation Study

In [ ]:
from training.ablation import run_ablation

# FUSeg: set to path if available, otherwise None (synthetic only)
FUSEG_DIR = 'data/fuseg' if Path('data/fuseg/images').exists() else None

results = run_ablation(
    fuseg_dir=FUSEG_DIR,
    synthetic_dir=SYNTHETIC_DIR,
    output_dir='results/ablation',
    batch_size=8,
    max_epochs=100,
    patience=10,
    lr=1e-4,
    num_points=64,
    pretrained=True,
)

## 4. Training Curves

In [ ]:
import json

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
decoder_names = ['polar', 'detr', 'autoregressive']
colors = ['#2563EB', '#16A34A', '#D97706']

for ax, name, color in zip(axes, decoder_names, colors):
    history_path = f'results/ablation/{name}/history.json'
    if Path(history_path).exists():
        with open(history_path) as f:
            h = json.load(f)
        epochs = range(1, len(h['train_loss']) + 1)
        ax.plot(epochs, h['train_loss'], color=color, alpha=0.5, label='Train')
        ax.plot(epochs, h['val_loss'], color=color, linewidth=2, label='Val')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss')
        ax.set_title(f'{name.capitalize()} Decoder')
        ax.legend()
        ax.grid(True, alpha=0.3)

plt.suptitle('Training Curves - Ablation Study', fontsize=14)
plt.tight_layout()
plt.savefig('figures/ablation_training_curves.png', dpi=150)
plt.show()

## 5. Polar Decoder Loss Components

In [ ]:
polar_history_path = 'results/ablation/polar/history.json'
if Path(polar_history_path).exists():
    with open(polar_history_path) as f:
        h = json.load(f)
    
    fig, ax = plt.subplots(figsize=(8, 4))
    epochs = range(1, len(h['loss_centroid']) + 1)
    ax.plot(epochs, h['loss_centroid'], label='L_centroid', color='#2563EB')
    ax.plot(epochs, h['loss_radii'], label='L_radii', color='#16A34A')
    ax.plot(epochs, h['loss_points'], label='L_points', color='#D97706')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss Component')
    ax.set_title('Polar Decoder - Loss Components')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('figures/polar_loss_components.png', dpi=150)
    plt.show()

## 6. Qualitative Comparison

In [ ]:
from models.encoder import CNNTransformerEncoder
from models.polar_decoder import PolarDecoder
from models.detr_decoder import DETRDecoder
from models.autoregressive_decoder import AutoregressiveDecoder
from data.dataset import WoundBoundaryDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load best checkpoints
models_loaded = {}
for name in ['polar', 'detr', 'autoregressive']:
    ckpt_path = f'results/ablation/{name}/best.pth'
    if Path(ckpt_path).exists():
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
        enc = CNNTransformerEncoder(pretrained=False).to(device)
        enc.load_state_dict(ckpt['encoder_state'])
        enc.eval()
        
        if name == 'polar':
            dec = PolarDecoder().to(device)
        elif name == 'detr':
            dec = DETRDecoder().to(device)
        else:
            dec = AutoregressiveDecoder().to(device)
        dec.load_state_dict(ckpt['decoder_state'])
        dec.eval()
        models_loaded[name] = (enc, dec)

print(f'Loaded {len(models_loaded)} models')

In [ ]:
# Qualitative grid
test_ds = WoundBoundaryDataset(synthetic_dir=SYNTHETIC_DIR, split='test', image_size=256, num_radii=64)
NUM_SHOW = 5

fig, axes = plt.subplots(NUM_SHOW, 4, figsize=(14, 3*NUM_SHOW))
col_titles = ['Ground Truth', 'Polar (ours)', 'DETR', 'Autoregressive']

for col, title in enumerate(col_titles):
    axes[0, col].set_title(title, fontsize=12, fontweight='bold')

np.random.seed(99)
indices = np.random.choice(len(test_ds), NUM_SHOW, replace=False)

for row, idx in enumerate(indices):
    sample = test_ds[idx]
    img = sample['image'].permute(1, 2, 0).numpy()
    gt = sample['points'].numpy() * 256
    gt_closed = np.vstack([gt, gt[0]])
    
    # GT
    axes[row, 0].imshow(img)
    axes[row, 0].plot(gt_closed[:, 0], gt_closed[:, 1], 'lime', linewidth=2)
    axes[row, 0].axis('off')
    
    # Predictions
    for col, name in enumerate(['polar', 'detr', 'autoregressive'], start=1):
        axes[row, col].imshow(img)
        axes[row, col].plot(gt_closed[:, 0], gt_closed[:, 1], 'lime', linewidth=1, alpha=0.4)
        
        if name in models_loaded:
            enc, dec = models_loaded[name]
            with torch.no_grad():
                feat = enc(sample['image'].unsqueeze(0).to(device))
                pred = dec(feat)
            pts = pred['points'][0].cpu().numpy() * 256
            pts_closed = np.vstack([pts, pts[0]])
            axes[row, col].plot(pts_closed[:, 0], pts_closed[:, 1], 'red', linewidth=2)
        axes[row, col].axis('off')

plt.suptitle('Qualitative Comparison: Predicted Boundaries', fontsize=14)
plt.tight_layout()
plt.savefig('figures/ablation_qualitative.png', dpi=150)
plt.show()

## 7. Results Summary Table

In [ ]:
import pandas as pd

if results:
    rows = []
    for name in ['polar', 'detr', 'autoregressive']:
        r = results[name]['eval']
        rows.append({
            'Decoder': name.capitalize(),
            'Chamfer (mm)': f"{r['chamfer']['mean']:.2f} +/- {r['chamfer']['std']:.2f}",
            'Hausdorff (mm)': f"{r['hausdorff']['mean']:.2f} +/- {r['hausdorff']['std']:.2f}",
            'IoU': f"{r['iou']['mean']:.3f} +/- {r['iou']['std']:.3f}",
            'Closure (mm)': f"{r['closure']['mean']:.3f} +/- {r['closure']['std']:.3f}",
            'Ordering (%)': f"{r['ordering']['mean']:.1f}",
            'Convergence': results[name]['convergence_epoch'],
        })
    
    df = pd.DataFrame(rows)
    print('\nAblation Study Results (Table 4.x for thesis):')
    print(df.to_string(index=False))
    df.to_csv('results/ablation/ablation_table.csv', index=False)
    print('\nSaved to results/ablation/ablation_table.csv')

## 8. Save All Figures List

Figures generated:
- `figures/synthetic_samples.png` — Fig. m1_qualitative
- `figures/ablation_training_curves.png` — Fig. m1_training
- `figures/polar_loss_components.png` — Loss component analysis
- `figures/ablation_qualitative.png` — Fig. ablation_comparison

In [ ]:
print('Done! All results saved.')
print('\nFiles to download from Kaggle:')
print('  results/ablation/polar/best.pth')
print('  results/ablation/detr/best.pth')
print('  results/ablation/autoregressive/best.pth')
print('  results/ablation/ablation_comparison.json')
print('  results/ablation/ablation_table.csv')
print('  figures/*.png')